In [ ]:
import io
import requests

from databricks.sdk import WorkspaceClient
from nyc_taxi_trips_etl.utils.pipeline_params_parsing import (
    parse_and_expand_year_months,
)

In [ ]:
# pipeline / run parameters
year_months_to_extract = dbutils.widgets.get("year_months_to_extract")

catalog = dbutils.widgets.get("catalog")
run_id = dbutils.widgets.get("run_id")
data_collection_started_at = dbutils.widgets.get("data_collection_started_at")

In [ ]:
# TODO: This goes into a config file or environment variables
endpoint_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/"
yellow_trips_file_prefix = "yellow_tripdata_"
yellow_trips_file_ext = ".parquet"

In [ ]:
year_months = parse_and_expand_year_months(year_months_to_extract)

In [ ]:
w = WorkspaceClient()

# TODO: Add error handling for failed downloads
#

for ym in year_months:
    file_name = f"{yellow_trips_file_prefix}{ym}{yellow_trips_file_ext}"
    print("Downloading file:", file_name)
    response = requests.get(f"{endpoint_url}{file_name}")

    if response.status_code == 200:
        dest_path = (
            f"/Volumes/{catalog}/landing/yellow_trip_data/{run_id}/{file_name}"
        )
        w.files.upload(dest_path, io.BytesIO(response.content), overwrite=True)
        print("File uploaded to Databricks workspace:", dest_path)
    else:
        print(f"Error downloading file '{file_name}':", response.text)
        # If we don't raise an exception here, the job will continue to run and
        # may fail later when trying to process the missing file.

200
